In [1]:
import pandas as pd
from re import match
from datetime import datetime
from dateutil.relativedelta import relativedelta
import locale

In [2]:
try:
    locale.setlocale(locale.LC_TIME, 'pt_BR.UTF-8')
except locale.Error:
    print("⚠️ Locale pt_BR.UTF-8 não disponível no sistema. Usando formato numérico.")

In [3]:
def name_Month(re_month:object) -> str:
    match_month:object = match(r"\d{4}-\d{2}-\d{2}", re_month)[0]

    month_raw:datetime = datetime(*list(map(int,match_month.split('-'))))

    month_str:str = month_raw.strftime("%B")

    return month_str.capitalize()


# Carga Prévia

In [38]:
df = pd.read_excel(r"C:\Users\BrunoKleinGomes\Documents\Jornada\Bases de Indicadores - Jornada de Segurança 2026 - Todas.xlsx", sheet_name="Previsto x realizado")

In [40]:
df.drop(df.loc[:, "Acumulado Prev":"Previsão mês próximo"], axis=1, inplace=True)

In [41]:
df.drop("Unnamed: 31", axis=1, inplace=True)

In [42]:
df.drop(df.query("Pilar == '-' or Pilar == 'TOTAL ACUMULADO' or Distribuidora == 'Global Distribuidora'").index,inplace=True)

In [43]:
nova_col = name_Month("2026-01-01 00:00:00.1")

In [44]:
nova_col_dict = {col: "Realizado {}/26".format(name_Month(col)) for col in df.loc[:, "2026-01-01 00:00:00.1":].columns}

In [45]:
df.rename(columns=nova_col_dict, inplace=True)

In [46]:
df.loc[:, :"Indicadores de resultado"].shape

(2342, 19)

In [47]:
nova_icol_dict = {col: "Previsto {}/26".format(col.strftime("%B").capitalize()) for col in df.iloc[:, 19:31].columns}

In [48]:
df.rename(columns=nova_icol_dict, inplace=True)

In [49]:
df_global = df.query("Regional == 'TODAS'")
df_regionais = df.query("Regional != 'TODAS'")

In [50]:
mask = df.columns.str.contains((datetime.now() - relativedelta(months=1)).strftime("%B").capitalize())
matching_columns = df.columns[mask]

In [36]:
tabela = df_global.query("`Status das ações` == 'Iniciada' and Distribuidora == 'PI' and Pilar != 'CAPACITAÇÃO' and `Ação Iniciada?` == 'Em Execução na Distribuidora'").groupby("Pilar").agg(
    Apuração=("Ação Iniciada?", "count"),
    Recebido=(matching_columns[0], "count")
)

In [35]:
tabela = df_global.query("`Status das ações` == 'Iniciada' and Distribuidora == '{}' and Pilar != 'CAPACITAÇÃO' and `Ação Iniciada?` == 'Em Execução na Distribuidora'".format()) \
.groupby("Pilar").agg(
    Apuração=("Ação Iniciada?", "count"),
    Recebido=("Realizado Abril/26", "count")
)

IndexError: Replacement index 0 out of range for positional args tuple

In [53]:
Distribuidoras = list(df_global['Distribuidora'].unique())

tabelas_distribuidoras = {}

for distribuidora in Distribuidoras:
    
    
    tabela_filtrada = df_global.query(
        "`Status das ações` == 'Iniciada' and "
        "Distribuidora == @distribuidora and "
        "Pilar != 'CAPACITAÇÃO' and "
        "`Ação Iniciada?` == 'Em Execução na Distribuidora'"
    ).groupby("Pilar").agg(
        Apuração=("Ação Iniciada?", "count"),
        Recebido=(matching_columns[1], "count")
    )
    

    tabelas_distribuidoras[distribuidora] = tabela_filtrada

In [54]:
for i,j in tabelas_distribuidoras.items():
    print(i)
    print(j)

AL
               Apuração  Recebido
Pilar                            
COMPORTAMENTO        10         0
FORNECEDOR            2         2
LIDERANÇA             6         6
POPULAÇÃO            12         0
AP
               Apuração  Recebido
Pilar                            
COMPORTAMENTO        13         0
FORNECEDOR            4         0
LIDERANÇA             6         6
POPULAÇÃO             8         0
GO
               Apuração  Recebido
Pilar                            
COMPORTAMENTO        13        13
FORNECEDOR            2         0
LIDERANÇA             6         6
POPULAÇÃO            14        12
MA
               Apuração  Recebido
Pilar                            
COMPORTAMENTO        11         5
FORNECEDOR            2         0
LIDERANÇA             6         4
POPULAÇÃO            12         0
PA
               Apuração  Recebido
Pilar                            
COMPORTAMENTO        11         0
FORNECEDOR            4         0
LIDERANÇA             6         0

In [16]:
(datetime.now() - relativedelta(months=1)).strftime("%B").capitalize()

'Maio'